# Large Batch and Layerwise Optimisers (C04 Topic 03)

**Citations:** Goyal et al. (2017 Linear Scaling); You, Gitman, Ginsburg (2017 LARS); McCandlish et al. (2018 Gradient Noise Scale); You et al. (2019 LAMB).

### Purpose & Topic Bridges
Distributed scale-out across thousands of accelerators requires scaling mini-batch sizes from hundreds to tens of thousands ($B \ge 32\text{k}$) to maximize compute utilization and amortize all-reduce communication latency.
However, naive batch scaling causes catastrophic divergence: applying the standard Linear Scaling Rule ($\eta \propto B$) causes gradients in certain layers to explode while others vanish.
**LARS** scales SGD updates using per-layer trust ratios $\phi_l = \frac{\|w_l\|_2}{\|g_l\|_2 + \lambda \|w_l\|_2 + \epsilon}$, while **LAMB** brings layerwise trust scaling to Adam-style adaptive moments $\phi_l = \frac{\|w_l\|_2}{\|r_l\|_2}$.

**Decision Boundary:** Deploy **LARS** for large-batch ($B \ge 8\text{k}$) vision/ConvNet training with SGD. Deploy **LAMB** for large-batch ($B \ge 16\text{k}$) Transformer/LLM pretraining. When NOT to use: Do NOT apply layerwise trust scaling to 1D parameters (LayerNorm gains or biases), as small norms will destabilize learning rates.

## 2. Mathematical Foundations & Governing Equations

1. **Linear Scaling Rule:** $\eta_B = \frac{B}{B_0} \eta_0$, causing catastrophic overshoot on layers with small $\frac{\|w_l\|_2}{\|g_l\|_2}$.
2. **LARS Layerwise Trust Ratio:**
$$\phi_l = \gamma \frac{\|w_l\|_2}{\|g_l\|_2 + \lambda \|w_l\|_2 + \epsilon}, \quad v_{t+1}^{(l)} = \mu v_t^{(l)} + \eta \phi_l (g_t^{(l)} + \lambda w_t^{(l)})$$
3. **LAMB Preconditioned Adam Step:**
$$r_t^{(l)} = \frac{\hat{m}_t^{(l)}}{\sqrt{\hat{v}_t^{(l)}} + \epsilon} + \lambda w_t^{(l)}, \quad \phi_l = \frac{\|w_t^{(l)}\|_2}{\|r_t^{(l)}\|_2}, \quad w_{t+1}^{(l)} = w_t^{(l)} - \eta \phi_l r_t^{(l)}$$
4. **Gradient Noise Scale ($B_{\text{simple}}$):**
$$B_{\text{simple}} = \frac{\text{Tr}(\Sigma)}{\|G\|_2^2} \approx \frac{\frac{1}{B-1}\sum_{b=1}^B \|g_b - \bar{g}\|_2^2}{\|\bar{g}\|_2^2}$$
5. **Computational Complexity:** LARS requires $\mathcal{O}(N)$ FLOPs and $\mathcal{O}(N)$ memory ($1\times$ momentum buffer). LAMB requires $\mathcal{O}(N)$ FLOPs and $2\times$ memory buffers ($m_t, v_t$) plus two scalar vector reductions per layer.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/04_training_methods_and_optimisers/01_gradient_and_curvature_optimisers/03_large_batch_and_layerwise_optimisers.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class LargeBatchOptimisersUnifiedModule(nn.Module):
    def __init__(self, in_dim=8, out_dim=8):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x): return self.linear(x)

    def step_lars(self, param, grad, v_buf, lr=0.1, momentum=0.9, weight_decay=0.0001, trust_coef=0.001, eps=1e-8):
        w_norm = torch.norm(param); g_norm = torch.norm(grad)
        if w_norm > 0 and g_norm > 0:
            trust_ratio = trust_coef * w_norm / (g_norm + weight_decay * w_norm + eps)
        else:
            trust_ratio = torch.tensor(1.0, device=param.device, dtype=param.dtype)
        local_lr = lr * trust_ratio; total_grad = grad + weight_decay * param
        new_v = momentum * v_buf + local_lr * total_grad
        return param - new_v, new_v, trust_ratio

    def step_lamb(self, param, grad, m_buf, v_buf, step=2, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-6, weight_decay=0.01):
        new_m = beta1 * m_buf + (1.0 - beta1) * grad
        new_v = beta2 * v_buf + (1.0 - beta2) * (grad * grad)
        m_hat = new_m / (1.0 - beta1**step); v_hat = new_v / (1.0 - beta2**step)
        adam_step = m_hat / (torch.sqrt(v_hat) + eps); r = adam_step + weight_decay * param
        w_norm = torch.norm(param); r_norm = torch.norm(r)
        if w_norm > 0 and r_norm > 0: trust_ratio = w_norm / r_norm
        else: trust_ratio = torch.tensor(1.0, device=param.device, dtype=param.dtype)
        return param - lr * trust_ratio * r, new_m, new_v, trust_ratio

    def compute_gradient_noise_scale(self, per_sample_grads):
        B, D = per_sample_grads.shape
        g_mean = torch.mean(per_sample_grads, dim=0); g_norm_sq = torch.sum(g_mean * g_mean)
        diff = per_sample_grads - g_mean.unsqueeze(0)
        trace_sigma = torch.sum(diff * diff) / float(B - 1)
        b_simple = trace_sigma / (g_norm_sq + 1e-12)
        return g_norm_sq, trace_sigma, b_simple

model = LargeBatchOptimisersUnifiedModule().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].to(device).t())
    model.linear.bias.copy_(weights['bias'].to(device))
print('LargeBatchOptimisersUnifiedModule initialized.')

In [ ]:
p_in = weights['param'].to(device); g_in = weights['grad'].to(device)
v_lars_in = weights['v_init_lars'].to(device)
m_lamb_in = weights['m_init_lamb'].to(device); v_lamb_in = weights['v_init_lamb'].to(device)
per_sample_grads = weights['per_sample_grads'].to(device)

# 1. LARS Verification
p_lars, v_lars, ratio_lars = model.step_lars(p_in, g_in, v_lars_in)
err_p_lars = (p_lars - weights['p_lars'].to(device)).abs().max().item()
err_v_lars = (v_lars - weights['v_lars'].to(device)).abs().max().item()
err_ratio_lars = (ratio_lars - weights['ratio_lars'].to(device)).abs().max().item()

# 2. LAMB Verification
p_lamb, m_lamb, v_lamb, ratio_lamb = model.step_lamb(p_in, g_in, m_lamb_in, v_lamb_in)
err_p_lamb = (p_lamb - weights['p_lamb'].to(device)).abs().max().item()
err_m_lamb = (m_lamb - weights['m_lamb'].to(device)).abs().max().item()
err_v_lamb = (v_lamb - weights['v_lamb'].to(device)).abs().max().item()
err_ratio_lamb = (ratio_lamb - weights['ratio_lamb'].to(device)).abs().max().item()

# 3. Gradient Noise Scale
g_sq, tr_sig, b_sim = model.compute_gradient_noise_scale(per_sample_grads)
err_g = (g_sq - weights['g_norm_sq'].to(device)).abs().max().item()
err_tr = (tr_sig - weights['trace_sigma'].to(device)).abs().max().item()
err_b = (b_sim - weights['b_simple'].to(device)).abs().max().item()

# 4. Linear Anchor
err_anchor = (model(test_in) - test_out).abs().max().item()

assert max(err_p_lars, err_v_lars, err_ratio_lars) < 1e-5, 'LARS parity failed'
assert max(err_p_lamb, err_m_lamb, err_v_lamb, err_ratio_lamb) < 1e-4, 'LAMB parity failed'
assert max(err_g, err_tr, err_b) < 1e-5, 'GNS parity failed'
assert err_anchor < 1e-5, 'Anchor parity failed'
print(f'All 4 invariants verified! LARS err: {err_p_lars:.2e}, LAMB err: {err_p_lamb:.2e}, GNS err: {err_b:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'LARS Step': lambda: model.step_lars(p_in, g_in, v_lars_in),
    'LAMB Step': lambda: model.step_lamb(p_in, g_in, m_lamb_in, v_lamb_in),
    'Gradient Noise Scale': lambda: model.compute_gradient_noise_scale(per_sample_grads),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=8)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Active Failure Mode: Naive Linear LR Scaling on Small/1D Parameters vs Trust Ratio Stabilization
# Scaling batch size from 256 to 32,768 (128x) with naive linear scaling sets lr = 128 * 0.001 = 0.128.
# On a small parameter (||w|| = 0.10) with unit update (||r|| = 1.0):
w_small = torch.tensor([0.10], device=device); r_grad = torch.tensor([1.0], device=device)
lr_scaled = 0.128  # 128x linear scaling

# 1. Naive update without layerwise trust clipping:
delta_naive = lr_scaled * r_grad
w_naive = w_small - delta_naive
relative_step_naive = float(torch.norm(delta_naive) / torch.norm(w_small))

# 2. LAMB layerwise trust ratio:
trust_ratio = float(torch.norm(w_small) / torch.norm(r_grad))
delta_lamb = lr_scaled * trust_ratio * r_grad
w_lamb = w_small - delta_lamb
relative_step_lamb = float(torch.norm(delta_lamb) / torch.norm(w_small))

print(f'[FAILURE DEMONSTRATED] Naive relative step: {relative_step_naive*100:.1f}% (parameter destroyed: {w_naive.item():.4f})')
print(f'[CORRECT] LAMB calibrated relative step: {relative_step_lamb*100:.1f}% (parameter stabilized: {w_lamb.item():.4f})')
assert relative_step_naive > 1.0, 'Naive update failed to exceed parameter magnitude'
assert relative_step_lamb <= lr_scaled + 1e-6, 'LAMB failed to constrain step relative to parameter norm'
print('[DIAGNOSTIC] Layerwise trust ratios prevent catastrophic gradient overshoot across multi-layer models.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Layerwise Trust Normalization:** Unconstrained linear scaling breaks when layers possess disparate condition numbers. LARS and LAMB dynamically scale updates so that the relative update $\frac{\|\Delta w_l\|}{\|w_l\|}$ is bounded.
2. **Critical Batch Size Diagnostic:** $B_{\text{simple}} = \frac{\text{Tr}(\Sigma)}{\|G\|_2^2}$ dictates the efficiency ceiling of data parallelism. Scaling batch size beyond $B_{\text{simple}}$ provides diminishing returns.
3. **Control Isomorphism:** Layerwise rate scaling functions as an Automatic Gain Control (AGC) loop across deep networks, preventing saturation and high-frequency oscillation in peripheral layers.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 02_modern_sign_and_factored_optimisers](file:///c:/Users/Liam/Code/AI-Examples/notebooks/04_training_methods_and_optimisers/01_gradient_and_curvature_optimisers/02_modern_sign_and_factored_optimisers.ipynb) | [Next: 04_orthogonal_and_preconditioned_matrix_optimisers](file:///c:/Users/Liam/Code/AI-Examples/notebooks/04_training_methods_and_optimisers/01_gradient_and_curvature_optimisers/04_orthogonal_and_preconditioned_matrix_optimisers.ipynb)